In [0]:
from typing import Any

In [0]:
import logging

from py4j.java_gateway import JavaObject  # type: ignore
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    concat_ws,
    current_timestamp,
    from_utc_timestamp,
    lit,
    sha2,
)

In [0]:
dbutils.widgets.text('curated_zone', 'None', '01-Curated_zone')
dbutils.widgets.text('catalog_landed', 'None', '02-Catalog_landed')
dbutils.widgets.text('reference_zone', 'None', '03-Reference_zone')
dbutils.widgets.text('run_date', 'None', '04-Run Date')
dbutils.widgets.text('schema', 'None', '05-Schema')
dbutils.widgets.text('env_prefix', 'None', '06-Environment')
curated_zone = dbutils.widgets.get('curated_zone')
catalog_landed = dbutils.widgets.get('catalog_landed')
reference_zone = dbutils.widgets.get('reference_zone')
run_date = dbutils.widgets.get('run_date')
schema = dbutils.widgets.get('schema')
env_prefix = dbutils.widgets.get('env_prefix')

import sys, os
relative_path = '../../helpers/lng_lnd'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from landing_helpers import *
from landed_helpers import *
relative_path = '../../helpers/curated_helpers'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from delta_table_creator import *
from extract import *
from transform import *
spark = SparkSession.builder.getOrCreate()


In [0]:
def load_curated_firm_structure_arotron(
    spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix
) -> None:
    args = [curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix]
    if all(item is not None for item in args) :
        date_filter = f"businessdate = {run_date!r}"

        # schema = (
        #     "FirmStructureId string,"
        #     "SourceSystem string,"
        #     "IngestionTimestamp timestamp,"
        #     "BatchId string,"
        #     "FirmStructureKey string,"
        #     "EmployeePartyKey string,"
        #     "EmployeeCategory string,"
        #     "EmployeeFirstName string,"
        #     "EmployeeLastName string,"
        #     "EmployeeFullName string,"
        #     "EmployeeEmail string,"
        #     "EmployeeUsername string,"
        #     "EmployeeUserTypeCode string,"
        #     "EmployeeUserType string,"
        #     "EmployeePositionCode string ,"
        #     "EmployeePosition string,"
        #     "DepartmentCode string,"
        #     "Department string,"
        #     "TeamCode string,"
        #     "Team string,"
        #     "SupervisorPartyKey string,"
        #     "SupervisorPartyName string,"
        #     "SupervisorUsername string,"
        #     "ManagerPartyKey string,"
        #     "ManagerPartyName string,"
        #     "ManagerUsername string,"
        #     "DirectorPartyKey string,"
        #     "DirectorPartyName string,"
        #     "DirectorUsername string,"
        #     "IsEmployeeActiveFlag boolean,"
        #     "CreatedDate date,"
        #     "LastUpdatedDate date,"
        #     "BusinessDate date"
        # )

        cluster_columns = ["BusinessDate"]

        curated_table_path = f"{curated_zone}.FirmStructure"

        # ================== Create Delta Table ==================
        logging.info("Creating Delta Table...")
        if spark.catalog.tableExists(curated_table_path) is False:
            create_delta_table(
                spark,
                curated_table_path,
                cluster_columns,
                schema,  # type: ignore
            )
        else:
            logging.info("Table already exists.")

        # ================== Delete From Delta Table ==================
        if spark.catalog.tableExists(curated_table_path) is True:
            delete_from_curated_table(
                spark,
                curated_table_path,
                run_date,  # type: ignore
                "arotron",
            )        
        else:
            logging.info("Tables does not exist, no need to delete entries for business date!")

        # ================== arxagent ==================

        arxagent_extra_filters = None

        if arxagent_extra_filters is not None:
            arxagent_extra_filters.append(date_filter)
            arxagent_final_filters = " AND ".join(arxagent_extra_filters)
        else:
            arxagent_final_filters = date_filter

        arxagent_selected_fields = [
            "UserId",
            "UserTeamID",
            "UserName",
            "businessdate",
        ]

        arxagent = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "arotron_arxagent",
            arxagent_final_filters,
            arxagent_selected_fields,
        )
        # ================== arxmdata ==================

        arxmdata_extra_filters = None

        if arxmdata_extra_filters is not None:
            arxmdata_extra_filters.append(date_filter)
            arxmdata_final_filters = " AND ".join(arxmdata_extra_filters)
        else:
            arxmdata_final_filters = date_filter

        arxmdata_selected_fields = [
            "LstItemRefCode",
            "LstItemDescr",
        ]

        arxmdata = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "arotron_arxmdata",
            arxmdata_final_filters,
            arxmdata_selected_fields,
        )

        # ================== firm_structure ==================
        firm_structure_arotron_primary_key_fields = [
            arxagent.UserId,
            lit("Employee"),
            lit("arotron"),
        ]
        FirmStructureKey_arotron_primary_key_fields = [
            arxagent.UserId,
            lit("Employee"),
        ]
        arxagent_arxmdata_join_condition = (
            arxagent["UserTeamID"] == arxmdata["LstItemRefCode"]
        )
        firm_structure_arotron = arxagent.join(
            arxmdata,
            arxagent_arxmdata_join_condition,
            "left",
        )
        firm_structure_arotron = (
            firm_structure_arotron.withColumn(
                "FirmStructureId",
                sha2(
                    concat_ws(";", *firm_structure_arotron_primary_key_fields),
                    256,
                ),
            )
        ).withColumn(
            "FirmStructureKey",
            concat_ws(";", *FirmStructureKey_arotron_primary_key_fields),
        )

        firm_structure_arotron = firm_structure_arotron.select(
            firm_structure_arotron.FirmStructureId.alias("FirmStructureId"),
            lit("arotron").alias("SourceSystem"),
            lit(from_utc_timestamp(current_timestamp(), "Europe/Athens")).alias(
                "IngestionTimestamp"
            ),
            lit(None).alias("BatchId"),
            firm_structure_arotron.FirmStructureKey.cast("string").alias(
                "FirmStructureKey"
            ),
            firm_structure_arotron.UserId.cast("string").alias("EmployeePartyKey"),
            lit("Employee").cast("string").alias("EmployeeCategory"),
            lit(None).cast("string").alias("EmployeeFirstName"),
            lit(None).cast("string").alias("EmployeeLastName"),
            firm_structure_arotron.UserName.cast("string").alias("EmployeeFullName"),
            lit(None).cast("string").alias("EmployeeEmail"),
            firm_structure_arotron.UserId.cast("string").alias("EmployeeUsername"),
            lit(None).cast("string").alias("EmployeeUserTypeCode"),
            lit(None).cast("string").alias("EmployeeUserType"),
            lit(None).cast("string").alias("EmployeePositionCode"),
            lit(None).cast("string").alias("EmployeePosition"),
            lit(None).cast("string").alias("DepartmentCode"),
            lit(None).cast("string").alias("Department"),
            firm_structure_arotron.UserTeamID.cast("string").alias("TeamCode"),
            firm_structure_arotron.LstItemDescr.cast("string").alias("Team"),
            lit(None).cast("string").alias("SupervisorPartyKey"),
            lit(None).cast("string").alias("SupervisorPartyName"),
            lit(None).cast("string").alias("SupervisorUsername"),
            lit(None).cast("string").alias("ManagerPartyKey"),
            lit(None).cast("string").alias("ManagerPartyName"),
            lit(None).cast("string").alias("ManagerUsername"),
            lit(None).cast("string").alias("DirectorPartyKey"),
            lit(None).cast("string").alias("DirectorPartyName"),
            lit(None).cast("string").alias("DirectorUsername"),
            lit(None).cast("boolean").alias("IsEmployeeActiveFlag"),
            lit(None).cast("date").alias("CreatedDate"),
            lit(None).cast("date").alias("LastUpdatedDate"),
            firm_structure_arotron.businessdate.cast("date").alias("BusinessDate"),
        )
        # ========================== Append ==========================
        firm_structure_arotron.write.format("delta").clusterBy("BusinessDate").mode(
            "append"
        ).saveAsTable(curated_table_path)
    else:
        null_args = [
            arg_name
            for arg_name, arg in args
            if arg
            in [
                curated_zone,
                catalog_landed,
                reference_zone,
                run_date,
                schema,
                env_prefix,
            ]
            and arg is None
        ]

        for arg_name in null_args:
            logging.error(f"No {arg_name} specified!")

    logging.info("We did it!")


In [0]:
load_curated_firm_structure_arotron(spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix)